In [6]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import ERA5Dataset
from data.dataloader import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

from huggingface_hub import snapshot_download

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


Load the configurations and the dataset.  

In [ ]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Config loaded!")

snapshot_download(repo_id=config["data"]["repo_id"],
                  repo_type='dataset',
                  local_dir=config["data"]["local_dir"], 
                  allow_patterns="*")

Config loaded!


Get the dataloaders for each split, ready to plug into the ML pipeline.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(config=config)

Using device: cuda
Creating datasets...
Dataset sizes -> Train: 87400, Val: 17488, Test: 17488
Creating dataloaders...
Dataloaders ready!


Test out the first iteration.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Fetch first batch
batch = next(iter(train_loader))  
batch = batch.to(device)          # move entire graph batch to GPU

print(batch)
print("x:", batch.x.shape, batch.x.device)
print("edge_index:", batch.edge_index.shape, batch.edge_index.device)
print("y:", batch.y.shape, batch.y.device)

DataBatch(x=[458752, 5], edge_index=[2, 2199552], y=[65536, 5], batch=[458752], ptr=[33])
x: torch.Size([458752, 5]) cuda:0
edge_index: torch.Size([2, 2199552]) cuda:0
y: torch.Size([65536, 5]) cuda:0
